# Projection cube and ECL

This notebook joins the monthly PD, LGD, EAD and discount-factor terms in the production calculation.

In [1]:
from pathlib import Path
import json
import sqlite3
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score, brier_score_loss, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd().resolve()
if not (ROOT / "config").exists():
    ROOT = ROOT.parent
DB = ROOT / "database" / "ifrs9_ecl.sqlite3"
CFG = yaml.safe_load((ROOT / "config" / "project.yaml").read_text())

def query(sql):
    with sqlite3.connect(DB) as connection:
        return pd.read_sql_query(sql, connection)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
plt.rcParams["figure.figsize"] = (9, 4)

Period ECL = Marginal PD x LGD x EAD x Discount Factor. Scenario ECL is summed first. Loan ECL is the weighted sum of complete scenario ECL values.

In [2]:
loan = query("select loan_id from worked_trace_summary where stage=2").iloc[0,0]
loan

'RM001506'

In [3]:
detail = query(f'''select loan_id,stage,scenario,scenario_weight,future_month,
conditional_pd,survival_probability,marginal_pd,cumulative_pd,
ead,projected_property_value,projected_ltv,lgd,discount_factor,period_ecl
from worked_trace_monthly where loan_id='{loan}' order by scenario,future_month''')
detail.head(24)

,loan_id,stage,scenario,scenario_weight,future_month,conditional_pd,survival_probability,marginal_pd,cumulative_pd,ead,projected_property_value,projected_ltv,lgd,discount_factor,period_ecl
0,RM001506,2,Base,0.5152,1,0.0372,1.0000,0.0372,0.0372,"230,343.7877","591,204.5654",38.9618,0.0426,0.9965,364.6185
1,RM001506,2,Base,0.5152,2,0.0372,0.9508,0.0354,0.0727,"230,067.8214","592,246.9165",38.8466,0.0426,0.9930,345.0283
2,RM001506,2,Base,0.5152,3,0.0372,0.9041,0.0337,0.1063,"229,790.8778","593,341.4980",38.7283,0.0426,0.9894,326.4098
3,RM001506,2,Base,0.5152,4,0.0372,0.8596,0.0320,0.1384,"229,512.9534","594,487.1598",38.6069,0.0426,0.9860,308.7218
4,RM001506,2,Base,0.5152,5,0.0372,0.8173,0.0304,0.1688,"229,234.0446","595,682.7966",38.4826,0.0426,0.9825,291.9242
5,RM001506,2,Base,0.5152,6,0.0372,0.7772,0.0289,0.1978,"228,954.1480","596,927.3454",38.3554,0.0425,0.9790,275.9782
6,RM001506,2,Base,0.5152,7,0.0372,0.7389,0.0275,0.2253,"228,673.2602","598,219.7838",38.2256,0.0425,0.9756,260.8460
7,RM001506,2,Base,0.5152,8,0.0372,0.7026,0.0262,0.2515,"228,391.3775","599,559.1284",38.0932,0.0424,0.9721,246.4911
8,RM001506,2,Base,0.5152,9,0.0379,0.6680,0.0253,0.2768,"228,108.4965","600,944.4329",37.9583,0.0424,0.9687,237.1127
9,RM001506,2,Base,0.5152,10,0.0423,0.6348,0.0269,0.3037,"227,824.6136","602,374.7866",37.8211,0.0423,0.9653,249.9056


In [4]:
detail['recalculated_ecl'] = detail.marginal_pd * detail.lgd * detail.ead * detail.discount_factor
(detail.period_ecl-detail.recalculated_ecl).abs().max()

np.float64(0.0)

In [5]:
detail.groupby(['scenario','scenario_weight'],as_index=False).period_ecl.sum()

,scenario,scenario_weight,period_ecl
0,Base,0.5152,"7,384.3776"
1,Downside,0.1672,"7,611.0877"
2,Upside,0.3176,"7,345.4070"


In [6]:
scenario_totals = detail.groupby(['scenario','scenario_weight'],as_index=False).period_ecl.sum()
scenario_totals['weighted_contribution'] = scenario_totals.period_ecl*scenario_totals.scenario_weight
scenario_totals[['scenario','period_ecl','scenario_weight','weighted_contribution']], scenario_totals.weighted_contribution.sum()

(   scenario  period_ecl  scenario_weight  weighted_contribution
 0      Base  7,384.3776           0.5152             3,804.1280
 1  Downside  7,611.0877           0.1672             1,272.8567
 2    Upside  7,345.4070           0.3176             2,332.9300,
 np.float64(7409.914751329568))

SQLite views provide separate Base, Upside and Downside outputs while ecl_projection_cube remains the authoritative calculation table.